# Residualization smoke test -- synthetic data

Exercises the full phenotype residualization pipeline (`../../scripts/local/residualize_lib.R`, the same code `residualize_phenotypes.ipynb` uses for real) against synthetic IDs, phenotypes, and PCs -- no AoU access needed. Two fake phenotypes: one generated near-normal (`fake_bmi`), one generated log-normal with injected outliers (`fake_triglycerides`), specifically so the diagnostics show whether the inverse-normal transform and outlier trimming are actually doing something.

## Setup

In [ ]:
library(dplyr)
library(readr)
source("../../scripts/local/residualize_lib.R")

set.seed(42)
OUT_DIR <- "/tmp/fake_pheno_out"
unlink(OUT_DIR, recursive = TRUE)

## Synthetic inputs

200 fake person IDs, fake sex-at-birth, fake PCs (written to a file and read back, same as a real `.eigenvec`), and fake zip3/SES joined via zip3 -- same join shape the real covariates use.

In [ ]:
N <- 200
keep_ids <- sprintf("SIM%04d", 1:N)

sex_lookup <- tibble(
  person_id = keep_ids,
  sex_at_birth = sample(c("Female", "Male"), N, replace = TRUE)
)

pc_cols <- paste0("PC", 1:20)
fake_pcs <- as.data.frame(matrix(rnorm(N * 20), nrow = N, ncol = 20))
names(fake_pcs) <- pc_cols
fake_pcs <- bind_cols(tibble(`#FID` = keep_ids, IID = keep_ids), fake_pcs)

pc_path <- "/tmp/fake_pcs.eigenvec"
write_tsv(fake_pcs, pc_path)

zip3_codes <- sprintf("%03d", sample(100:999, 15))
person_zip3 <- tibble(person_id = keep_ids, zip3 = sample(zip3_codes, N, replace = TRUE))
ses_by_zip3 <- tibble(
  zip3 = zip3_codes,
  median_income = round(rnorm(15, 60000, 15000)),
  poverty = pmax(0, pmin(1, rnorm(15, 0.12, 0.05))),
  deprivation_index = rnorm(15, 0, 1)
)

## Fake phenotype list + pull functions

`fake_bmi`: roughly normal, no outliers. `fake_triglycerides`: log-normal (right-skewed) with 5 injected large outliers -- deliberately picked so the diagnostics below show a real before/after difference, not just near-zero skew everywhere.

In [ ]:
pheno_list <- tibble(
  phenotype_name = c("fake_bmi", "fake_triglycerides"),
  source = c("measurement", "measurement"),
  concept_id = c("0", "0")
)

pull_phenotype_fake <- function(row, keep_ids) {
  n <- length(keep_ids)
  sex <- sex_lookup$sex_at_birth[match(keep_ids, sex_lookup$person_id)]
  age <- round(runif(n, 18, 80))
  sex_effect <- ifelse(sex == "Female", 2, -2)

  if (row$phenotype_name == "fake_bmi") {
    pheno <- 25 + sex_effect + 0.05 * age + rnorm(n, sd = 4)
  } else {
    pheno <- exp(rnorm(n, mean = 4.5, sd = 0.4)) + 0.3 * age
    outlier_idx <- sample(n, 5)
    pheno[outlier_idx] <- pheno[outlier_idx] + 400
  }

  tibble(person_id = keep_ids, phenotype = pheno, age = age, sex_at_birth = sex)
}

pull_covariates_fake <- function(keep_ids) {
  pcs <- read_tsv(pc_path, show_col_types = FALSE) %>%
    rename(person_id = IID) %>%
    filter(person_id %in% keep_ids) %>%
    select(-`#FID`)

  zip3 <- person_zip3 %>% filter(person_id %in% keep_ids)
  ses <- zip3 %>% left_join(ses_by_zip3, by = "zip3") %>% select(-zip3)

  list(pcs = pcs, zip3 = zip3, ses = ses)
}

## Run

Same `run_residualization()` the real notebook calls -- full cross product of {phenotype} x {raw, invnorm} x {covariate-set}, one `.pheno` file per combination.

In [ ]:
result <- run_residualization(
  pheno_list, keep_ids, pull_phenotype_fake, pull_covariates_fake,
  build_covariate_sets(pc_cols), OUT_DIR
)

## Diagnostics

`fake_bmi` skewness should stay near 0 either way (generated near-normal). `fake_triglycerides` raw skewness should be clearly positive, dropping to ~0 after the invnorm transform -- if it doesn't, something's wrong with the transform, not the fake data.

In [ ]:
result$skew_summary_table

In [ ]:
result$combo_summary_table

## Spot-check the output files

In [ ]:
list.files(OUT_DIR)

In [ ]:
read.table(file.path(OUT_DIR, "fake_triglycerides__invnorm__base_pcs.pheno"), header = TRUE) %>% head()